# ProverbGap: Shortcut Audit Fix (Pilot v4 — Full Grid)
This notebook implements a full multi-model, multi-strategy grid search to rigorously test which prompt configuration successfully eliminates shortcuts.

**Grid:**
- Tasks: Task A (Literal) & Task B (Cultural)
- Strategies & Models:
  1. `zs` (Zero-Shot) → Cerebras (Llama 3.1 8B)
  2. `cot_en` (Chain-of-Thought English) → Groq (Llama 3.3 70B)
  3. `cot_xl` (Chain-of-Thought Cross-Lingual) → Groq (Llama 3.1 8B)
- Languages: English, Yoruba, Arabic (N=70 each)


In [ ]:
# Cell 1: Setup and API Key Loading
import subprocess, sys, time, os, json, re, random
from pathlib import Path
!pip install -q pandas requests
import pandas as pd
import requests

def _load_secrets():
    keys = {}
    try:
        from kaggle_secrets import UserSecretsClient
        s = UserSecretsClient()
        def get(name):
            try: 
                val = s.get_secret(name)
                return val if val and val.strip() else None
            except: return None
            
        keys['GROQ'] = [get(f'GROQ_API_KEY{"_"+str(i) if i>1 else ""}') for i in range(1,9)]
        keys['GROQ'] = [k for k in keys['GROQ'] if k]
        
        keys['CEREBRAS'] = [get(f'CEREBRAS_API_KEY{"_"+str(i) if i>1 else ""}') for i in range(1,3)]
        keys['CEREBRAS'] = [k for k in keys['CEREBRAS'] if k]
    except ImportError:
        print("Not running on Kaggle. Add keys to environment variables.")
        keys['GROQ'] = [os.environ.get('GROQ_API_KEY')] if os.environ.get('GROQ_API_KEY') else []
        keys['CEREBRAS'] = [os.environ.get('CEREBRAS_API_KEY')] if os.environ.get('CEREBRAS_API_KEY') else []
        
    return keys

KEYS = _load_secrets()
print(f"Loaded Keys:")
for k, v in KEYS.items():
    print(f"  {k}: {len(v)} keys loaded")

OUT_DIR = Path('/kaggle/working')
OUT_DIR.mkdir(exist_ok=True)
print("Setup complete.")


In [ ]:
# Cell 2: Prompts (Zero-Shot, CoT-EN, CoT-XL for Tasks A & B)
import re, json

# --- TASK B (CULTURAL) ---
SYS_B_BASE = (
    'You are building a rigorous benchmark to test cultural understanding of proverbs. '
    'Given a proverb and its correct cultural meaning, you must generate 3 ALTERNATIVE cultural '
    'interpretations that act as distractors.\n\n'
    'CRITICAL INSTRUCTION: Your distractors must be EXACT STRUCTURAL CLONES of the correct meaning. '
    'If the correct meaning is an archaic proverb, your distractors MUST be archaic proverbs. '
    'If it is a modern explanation starting with "To...", your distractors MUST start with "To...". '
    'If it uses highly formal vocabulary, you must do the same. They must be indistinguishable in style and length.\n\n'
)

SYS_B_ZS = SYS_B_BASE + 'Return ONLY a JSON array of 3 strings matching the structure: ["distractor 1", "distractor 2", "distractor 3"]. No explanation.'

SYS_B_COT_EN = SYS_B_BASE + (
    'Reason step-by-step IN ENGLISH to plan your structural mimicry. You MUST output valid JSON exactly matching this format:\n'
    '{\n'
    '  "reasoning": "Step 1: Analyze structure. Step 2: Plan distractors",\n'
    '  "distractors": ["distractor 1", "distractor 2", "distractor 3"]\n'
    '}'
)

SYS_B_COT_XL = SYS_B_BASE + (
    'Reason step-by-step IN THE NATIVE LANGUAGE OF THE PROVERB to plan your structural mimicry. You MUST output valid JSON exactly matching this format:\n'
    '{\n'
    '  "reasoning": "Step 1: Analyze structure natively. Step 2: Plan distractors",\n'
    '  "distractors": ["English distractor 1", "English distractor 2", "English distractor 3"]\n'
    '}'
)

# --- TASK A (LITERAL) ---
SYS_A_BASE = (
    'You are building a rigorous benchmark to test literal comprehension of proverbs. '
    'Given a proverb and its correct English translation, you must generate 3 ALTERNATIVE English '
    'translations that act as distractors.\n\n'
    'CRITICAL INSTRUCTION: Your distractors must be EXACT STRUCTURAL CLONES of the correct translation. '
    'If the correct translation uses archaic words, your distractors MUST use archaic words. '
    'They must be indistinguishable in grammar, style, and length.\n\n'
)

SYS_A_ZS = SYS_A_BASE + 'Return ONLY a JSON array of 3 strings matching the structure: ["distractor 1", "distractor 2", "distractor 3"]. No explanation.'

SYS_A_COT_EN = SYS_A_BASE + (
    'Reason step-by-step IN ENGLISH to plan your structural mimicry. You MUST output valid JSON exactly matching this format:\n'
    '{\n'
    '  "reasoning": "Step 1: Analyze structure. Step 2: Plan distractors",\n'
    '  "distractors": ["distractor 1", "distractor 2", "distractor 3"]\n'
    '}'
)

SYS_A_COT_XL = SYS_A_BASE + (
    'Reason step-by-step IN THE NATIVE LANGUAGE OF THE PROVERB to plan your structural mimicry. You MUST output valid JSON exactly matching this format:\n'
    '{\n'
    '  "reasoning": "Step 1: Analyze structure natively. Step 2: Plan distractors",\n'
    '  "distractors": ["English distractor 1", "English distractor 2", "English distractor 3"]\n'
    '}'
)

# -------------------------

def make_prompt(task, proverb, translation, correct, lang):
    if task == 'b':
        yoruba_note = ''
        if lang == 'Yoruba':
            yoruba_note = '\nIMPORTANT: Reflect specific Yoruba social values or communal principles.'
        return f'Proverb ({lang}): {proverb}\nEnglish translation: {translation}\nCorrect cultural meaning: {correct}{yoruba_note}\n\nReturn ONLY the JSON. No markdown.'
    else:
        return f'Proverb ({lang}): {proverb}\nCorrect English translation: {translation}\n\nReturn ONLY the JSON. No markdown.'

def get_sys_prompt(task, strategy):
    if task == 'b':
        if strategy == 'zs': return SYS_B_ZS
        if strategy == 'cot_en': return SYS_B_COT_EN
        if strategy == 'cot_xl': return SYS_B_COT_XL
    else:
        if strategy == 'zs': return SYS_A_ZS
        if strategy == 'cot_en': return SYS_A_COT_EN
        if strategy == 'cot_xl': return SYS_A_COT_XL

def parse_response(raw):
    if not raw: return None
    raw = re.sub(r'<reasoning>.*?</reasoning>', '', raw, flags=re.DOTALL)
    raw = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL)
    raw = raw.strip()
    raw = re.sub(r'^```\w*\n?', '', raw, flags=re.MULTILINE)
    raw = re.sub(r'\n?```$', '', raw, flags=re.MULTILINE).strip()
    
    try:
        d = json.loads(raw)
        if isinstance(d, dict) and 'distractors' in d and len(d['distractors']) >= 3:
            return [str(x).strip() for x in d['distractors'][:3] if str(x).strip()]
        elif isinstance(d, list) and len(d) >= 3:
            return [str(x).strip() for x in d[:3] if str(x).strip()]
    except: pass
    
    m = re.search(r'\{[\s\S]*\}', raw)
    if m:
        try:
            d = json.loads(m.group())
            if isinstance(d, dict) and 'distractors' in d and len(d['distractors']) >= 3:
                return [str(x).strip() for x in d['distractors'][:3] if str(x).strip()]
        except: pass
        
    m2 = re.search(r'\[[\s\S]*?\]', raw)
    if m2:
        try:
            d = json.loads(m2.group())
            if isinstance(d, list) and len(d) >= 3:
                return [str(x).strip() for x in d[:3] if str(x).strip()]
        except: pass
        
    return None

def assemble_mcq(correct, distractors):
    rng = random.Random(42)
    choices = list(distractors[:3]) + [correct]
    rng.shuffle(choices)
    labels = ['A', 'B', 'C', 'D']
    answer = labels[choices.index(correct)]
    choice_dict = {f'Choice_{l}': choices[i] for i, l in enumerate(labels)}
    return choice_dict, answer


In [ ]:
# Cell 3: Strict Smart API Callers (Mapped to specific models)
_key_indices = {k: 0 for k in KEYS}

def get_next_key(provider):
    if not KEYS.get(provider): return None
    k = KEYS[provider][_key_indices[provider] % len(KEYS[provider])]
    _key_indices[provider] += 1
    return k

def call_groq(sys_prompt, user_prompt, model, temp=0.85, max_tokens=1200):
    for attempt in range(4):
        key = get_next_key('GROQ')
        if not key: return None
        try:
            r = requests.post('https://api.groq.com/openai/v1/chat/completions', headers={
                'Authorization': f'Bearer {key}'
            }, json={
                'model': model,
                'messages': [{'role':'system','content':sys_prompt}, {'role':'user','content':user_prompt}],
                'temperature': temp, 'max_tokens': max_tokens
            }, timeout=30)
            if r.status_code == 200:
                return r.json()['choices'][0]['message']['content']
            elif r.status_code == 429:
                time.sleep(2)
        except Exception as e:
            time.sleep(1)
    return None

def call_cerebras(sys_prompt, user_prompt, model='llama3.1-8b', temp=0.85, max_tokens=1200):
    for attempt in range(4):
        key = get_next_key('CEREBRAS')
        if not key: return None
        try:
            r = requests.post('https://api.cerebras.ai/v1/chat/completions', headers={
                'Authorization': f'Bearer {key}'
            }, json={
                'model': model,
                'messages': [{'role':'system','content':sys_prompt}, {'role':'user','content':user_prompt}],
                'temperature': temp, 'max_tokens': max_tokens
            }, timeout=30)
            if r.status_code == 200:
                return r.json()['choices'][0]['message']['content']
            elif r.status_code == 429:
                time.sleep(4)
        except Exception as e:
            time.sleep(1)
    return None

def generate_distractors(sys_prompt, user_prompt, strategy):
    # EXACT MODEL ROUTING BASED ON ORIGINAL NOTEBOOK:
    if strategy == 'zs':
        # ZS uses Cerebras
        return call_cerebras(sys_prompt, user_prompt, model='llama3.1-8b')
    elif strategy == 'cot_en':
        # CoT English uses Groq 70B
        return call_groq(sys_prompt, user_prompt, model='llama-3.3-70b-versatile')
    elif strategy == 'cot_xl':
        # CoT Cross-Lingual uses Groq 8B
        return call_groq(sys_prompt, user_prompt, model='llama-3.1-8b-instant')
    return None

def run_audit(sys_prompt, user_prompt):
    res = call_groq(sys_prompt, user_prompt, model='llama-3.1-8b-instant', temp=0.0, max_tokens=5)
    if res: return res
    res = call_cerebras(sys_prompt, user_prompt, model='llama3.1-8b', temp=0.0, max_tokens=5)
    return res


In [ ]:
# Cell 4: Load Pilot Data
import pandas as pd
from pathlib import Path

PILOT_N = 70
LANGUAGES = ['English', 'Yoruba', 'Arabic']

dfs = {}
input_dir = Path('/kaggle/input')
if not input_dir.exists():
    input_dir = Path('.')

for lang in LANGUAGES:
    filename = f'{lang}_cleaned.csv'
    found_files = list(input_dir.rglob(filename))
    
    if found_files:
        csv_path = found_files[0]
        df = pd.read_csv(csv_path)
        
        col_map = {
            'Source_Text_Yo': 'source_proverb', 'Source_Text_Mid': 'source_proverb', 'Proverb': 'source_proverb', 'source_text': 'source_proverb',
            'Target_Text_En': 'proverb_en', 'Translation': 'proverb_en', 'english_translation': 'proverb_en',
            'Cultural_Context': 'correct_meaning', 'Correct_Meaning': 'correct_meaning'
        }
        df = df.rename(columns=col_map)
        
        if lang == 'English':
            if 'proverb_en' not in df.columns and 'source_proverb' in df.columns:
                df['proverb_en'] = df['source_proverb']
            if 'correct_meaning' not in df.columns and 'proverb_en' in df.columns:
                df['correct_meaning'] = df['proverb_en']
                
        required_cols = ['source_proverb', 'proverb_en', 'correct_meaning']
        if all(c in df.columns for c in required_cols):
            df = df.dropna(subset=required_cols).reset_index(drop=True)
            if 'Sample_ID' not in df.columns and 'sample_id' not in df.columns:
                df['sample_id'] = [f'{lang[:3].upper()}{str(i+1).zfill(4)}' for i in range(len(df))]
            
            dfs[lang] = df.sample(min(PILOT_N, len(df)), random_state=42)
            print(f"✅ {lang}: Loaded {len(dfs[lang])} items")
        else:
            missing = [c for c in required_cols if c not in df.columns]
            print(f"❌ {lang}: Missing columns: {missing}")
    else:
        print(f"❌ {lang}: File {filename} not found")


In [ ]:
# Cell 5: Master Multi-Model Grid Execution Loop (Task A & B x 3 Strategies)
TASKS = ['a', 'b']
STRATEGIES = ['zs', 'cot_en', 'cot_xl']

overall_results = []

for PILOT_TASK in TASKS:
    task_name = "Literal (Task A)" if PILOT_TASK == 'a' else "Cultural (Task B)"
    
    for STRATEGY in STRATEGIES:
        print(f"\n{'='*70}\nSTARTING {task_name.upper()} — STRATEGY: {STRATEGY.upper()}\n{'='*70}")
        
        gen_rows = []
        sys_p = get_sys_prompt(PILOT_TASK, STRATEGY)
        
        # 1. GENERATION PHASE
        for lang, df in dfs.items():
            print(f"[{lang}] Generating...")
            fail_count = 0
            for idx, row in df.iterrows():
                proverb = str(row['source_proverb'])
                translation = str(row['proverb_en'])
                correct = str(row['correct_meaning']) if PILOT_TASK == 'b' else translation
                
                user_p = make_prompt(PILOT_TASK, proverb, translation, correct, lang)
                raw = generate_distractors(sys_p, user_p, STRATEGY)
                dists = parse_response(raw)
                
                if not dists:
                    fail_count += 1
                    continue
                    
                choices, answer = assemble_mcq(correct, dists)
                gen_rows.append({
                    'language': lang,
                    'sample_id': str(row.get('sample_id', f'{lang[:3]}{idx:04d}')),
                    'source_proverb': proverb,
                    'proverb_en': translation,
                    'correct_meaning': correct,
                    **choices,
                    'Answer': answer,
                    'distractor_1': dists[0],
                    'distractor_2': dists[1],
                    'distractor_3': dists[2],
                })
            print(f"  Done {lang}. Generated: {len(gen_rows)} total. Fails: {fail_count}")
                    
        gen_df = pd.DataFrame(gen_rows)
        if len(gen_df) == 0:
            print("❌ No data generated. Skipping audit.")
            continue
            
        gen_path = OUT_DIR / f'pilot_v4_gen_{PILOT_TASK}_{STRATEGY}.csv'
        gen_df.to_csv(gen_path, index=False)
            
        # 2. AUDIT PHASE
        print(f'\n=== BLIND SHORTCUT AUDIT ({STRATEGY.upper()}) ===')
        AUDIT_SYS = (
            'You are answering a multiple-choice question about a proverb. '
            'Select the option that best matches the question. '
            'Reply with ONLY the letter: A, B, C, or D.'
        )
        AUDIT_Q = {
            'a': 'What does this proverb most likely mean literally?',
            'b': 'Which option best captures the cultural meaning of this proverb?',
        }

        def audit_prompt(task, proverb, translation, ca, cb, cc, cd):
            return (
                f'Proverb: {proverb}\nEnglish: {translation}\n\n'
                f'{AUDIT_Q[task]}\nA. {ca}\nB. {cb}\nC. {cc}\nD. {cd}\n\nAnswer (A/B/C/D only):'
            )

        audit_rows = []
        for i, row in gen_df.iterrows():
            ap = audit_prompt(
                PILOT_TASK, row['source_proverb'], row['proverb_en'],
                row['Choice_A'], row['Choice_B'], row['Choice_C'], row['Choice_D']
            )
            raw = run_audit(AUDIT_SYS, ap)
            raw = (raw or '').strip().upper()
            pred = None
            for ch in 'ABCD':
                if raw.startswith(ch): pred = ch; break
            if not pred:
                m = re.search(r'\b([ABCD])\b', raw)
                pred = m.group(1) if m else None
                
            audit_rows.append({
                'language': row['language'],
                'sample_id': row['sample_id'],
                'correct': row['Answer'],
                'predicted': pred,
                'is_correct': int(pred == row['Answer']) if pred else 0,
            })

        audit_df = pd.DataFrame(audit_rows)
        audit_path = OUT_DIR / f'pilot_v4_audit_{PILOT_TASK}_{STRATEGY}.csv'
        audit_df.to_csv(audit_path, index=False)

        print(f'\n--- SRS SCOREBOARD ({task_name} - {STRATEGY}) ---')
        print(f'{"Language":<12} {"N":>5} {"Acc% (SRS)":>12} {"Status":>12}')
        print('-'*45)
        for lang in LANGUAGES:
            sub = audit_df[audit_df['language']==lang]
            if len(sub)==0: continue
            acc = round(sub['is_correct'].mean()*100, 1)
            status = '✅ PASS' if acc < 40 else ('🟡 REVIEW' if acc < 55 else '🔴 FAIL')
            print(f'{lang:<12} {len(sub):>5} {acc:>10.1f}% {status:>12}')
            overall_results.append({'Task': PILOT_TASK, 'Strategy': STRATEGY, 'Lang': lang, 'SRS': acc})

        overall = round(audit_df['is_correct'].mean()*100, 1)
        print('-'*45)
        print(f'{"OVERALL":<12} {len(audit_df):>5} {overall:>10.1f}%')
        
# 3. FINAL SUMMARY
print("\n\n" + "*"*50)
print("               FINAL EXPERIMENT SUMMARY")
print("*"*50)
sum_df = pd.DataFrame(overall_results)
pivot = sum_df.pivot(index=['Task', 'Strategy'], columns='Lang', values='SRS')
print(pivot.to_string())
